In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#!ls "/content/drive/MyDrive/Colab Notebooks/"

In [ ]:
import pandas as pd
import joblib
import numpy as np

FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/"

# ===============================
# LOAD MODEL
# ===============================
package = joblib.load(FILE_PATH + "windows_threat_model.pkl")

model = package["model"]
features = package["features"]

# ===============================
# FEATURE ENGINEERING
# ===============================
def feature_engineering(df):

    df.columns = df.columns.str.strip().str.replace(" ", "_").str.lower()

    df["event_id"] = pd.to_numeric(df["event_id"], errors="coerce").fillna(0)

    df["date_and_time"] = pd.to_datetime(
        df["date_and_time"],
        format="%d.%m.%Y. %H:%M:%S",
        errors="coerce"
    )

    df = df.sort_values("date_and_time")

    df["hour"] = df["date_and_time"].dt.hour.fillna(0)

    df["night_activity"] = ((df["hour"] < 5) | (df["hour"] > 23)).astype(int)

    # Better frequency signal
    df["event_freq"] = df.groupby("event_id").cumcount()

    df["burst_count"] = df["event_freq"].clip(upper=10)

    # Time difference feature
    df["time_diff"] = df["date_and_time"].diff().dt.total_seconds().fillna(0)

    return df


# ===============================
# LOAD DATA
# ===============================
df = pd.read_csv(FILE_PATH + "uploaded_logs.csv")

df = feature_engineering(df)

print("Logs loaded:", len(df))

# ===============================
# PREPARE FEATURES
# ===============================
X = df[features]

# Ensure correct column order
if hasattr(model, "feature_names_in_"):
    X = X[model.feature_names_in_]

# ===============================
# PREDICT
# ===============================
probs = model.predict_proba(X)[:, 1]

# ===============================
# DYNAMIC THRESHOLD
# ===============================
t1 = np.percentile(probs, 90)
t2 = np.percentile(probs, 98)

df["risk_score"] = 0
df.loc[probs > t1, "risk_score"] = 1
df.loc[probs > t2, "risk_score"] = 2

# ===============================
# DEBUG
# ===============================
print("\n=== PROBABILITY DEBUG ===")
print("Max:", probs.max())
print("Mean:", probs.mean())

print("\nRisk distribution")
print(df["risk_score"].value_counts())

# ===============================
# SAVE
# ===============================
df.to_csv(FILE_PATH + "windows_ai_output.csv", index=False)

print("AI detection completed")

Logs loaded: 1000

=== PROBABILITY DEBUG ===
Max: 0.5626296129452601
Mean: 0.151874600823774

Risk distribution
risk_score
0    900
1     80
2     20
Name: count, dtype: int64
AI detection completed


In [ ]:
# # ===============================
# # DETECTION (FIXED)
# # ===============================
# import pandas as pd
# import joblib
# import numpy as np

# FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/"

# # ===============================
# # LOAD MODEL
# # ===============================
# package = joblib.load(FILE_PATH + "windows_threat_model.pkl")

# model = package["model"]
# features = package["features"]

# # ===============================
# # FEATURE ENGINEERING
# # ===============================
# def feature_engineering(df):

#     df.columns = df.columns.str.strip().str.replace(" ", "_").str.lower()

#     # ✅ FIX 1: Ensure event_id numeric
#     df["event_id"] = pd.to_numeric(df["event_id"], errors="coerce").fillna(0)

#     df["date_and_time"] = pd.to_datetime(
#         df["date_and_time"],
#         format="%d.%m.%Y. %H:%M:%S",
#         errors="coerce"
#     )

#     df = df.sort_values("date_and_time")

#     df["hour"] = df["date_and_time"].dt.hour.fillna(0)

#     df["night_activity"] = ((df["hour"] < 6) | (df["hour"] > 22)).astype(int)

#     df["event_freq"] = df.groupby("event_id").cumcount()

#     df["burst_count"] = df.groupby("event_id").cumcount().clip(upper=20)

#     return df

# # ===============================
# # LOAD DATA
# # ===============================
# df = pd.read_csv(FILE_PATH + "uploaded_logs.csv")

# df = feature_engineering(df)

# print("Logs loaded:", len(df))

# # ===============================
# # PREDICTION
# # ===============================
# X = df[features]

# probs = model.predict_proba(X)[:, 1]

# # ===============================
# # DEBUG (IMPORTANT)
# # ===============================
# print("\n=== PROBABILITY DEBUG ===")
# print("Max probability:", probs.max())
# print("Mean probability:", probs.mean())

# # ===============================
# # THRESHOLD (FIXED)
# # ===============================
# df["risk_score"] = 0

# df.loc[probs > 0.35, "risk_score"] = 1   # LOWERED
# df.loc[probs > 0.70, "risk_score"] = 2   # LOWERED

# # ===============================
# # DISTRIBUTION
# # ===============================
# print("\nRisk distribution")
# print(df["risk_score"].value_counts())

# # ===============================
# # SAVE OUTPUT
# # ===============================
# df.to_csv(FILE_PATH + "windows_ai_output.csv", index=False)

# print("AI detection completed")

Logs loaded: 1000

=== PROBABILITY DEBUG ===
Max probability: 1.0
Mean probability: 0.1660657803846784

Risk distribution
risk_score
0    856
2    114
1     30
Name: count, dtype: int64
AI detection completed


In [ ]:
# !ls "/content/drive/MyDrive/Colab Notebooks/"

In [ ]:
# import pandas as pd
# import joblib

# from sklearn.model_selection import train_test_split
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report
# from sklearn.metrics import confusion_matrix
# from sklearn.metrics import accuracy_score

# FILE_PATH="/content/drive/MyDrive/Colab Notebooks/"

# df=pd.read_csv(FILE_PATH+"windows_processed_logs.csv")

# print("Logs loaded:",len(df))

# # ==============================
# # FEATURES
# # ==============================

# features=[

# "Event_ID",
# "hour_sin",
# "hour_cos",
# "night_activity",
# "event_freq",
# "event_freq_log",
# "burst_count",
# "is_privileged_event",
# "event_rarity"

# ]

# X=df[features]

# y=df["label"]

# # ==============================
# # TRAIN TEST SPLIT
# # ==============================

# X_train,X_test,y_train,y_test=train_test_split(

# X,
# y,
# test_size=0.2,
# random_state=42,
# stratify=y

# )

# # ==============================
# # STRICT RANDOM FOREST
# # ==============================

# model=RandomForestClassifier(

# n_estimators=600,
# max_depth=20,
# min_samples_split=2,
# min_samples_leaf=1,
# random_state=42,
# n_jobs=-1

# )

# model.fit(X_train,y_train)

# # ==============================
# # PREDICTIONS
# # ==============================

# y_pred=model.predict(X_test)

# # ==============================
# # EVALUATION
# # ==============================

# print("\nAccuracy:",accuracy_score(y_test,y_pred))

# print("\nClassification Report")
# print(classification_report(y_test,y_pred))

# print("\nConfusion Matrix")
# print(confusion_matrix(y_test,y_pred))

# # ==============================
# # SAVE MODEL
# # ==============================
# package = {
#     "model": model,
#     "features": features
# }

# joblib.dump(package, FILE_PATH+"windows_threat_detection_model.pkl")

# print("\nModel saved successfully")

Logs loaded: 84065

Accuracy: 0.9550347945042527

Classification Report
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      5098
           1       0.92      0.88      0.90      2245
           2       0.96      0.96      0.96      5597
           3       0.96      0.97      0.96      3873

    accuracy                           0.96     16813
   macro avg       0.95      0.95      0.95     16813
weighted avg       0.95      0.96      0.95     16813


Confusion Matrix
[[4993   85   20    0]
 [ 162 1974   97   12]
 [  32   72 5349  144]
 [   0   10  122 3741]]

Model saved successfully


In [ ]:
# from google.colab import drive
# import os

# # Unmount Google Drive
# drive.flush_and_unmount()
# print("Drive unmounted")

# # Terminate runtime session
# os.kill(os.getpid(), 9)